In [30]:
%pip install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [46]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from tqdm import tqdm
import folium
from folium.plugins import MarkerCluster

In [32]:
data = pd.read_excel('../data/raw/active_points_Other.xlsx', engine="openpyxl", skiprows=9, header=None,
    names=["Тип точки", "Регион", "Город", "address_raw", "Техническое название", "working_hours"])

In [33]:
data = data[data["Город"] == "Красноярск"].reset_index(drop=True)
data['id'] = range(len(data))

data = data[["address_raw", "working_hours"]]

data['address_clean'] = data['address_raw'].copy()

In [34]:
expressions = {"  ": " ", "Россия": "", "Край": "", "Красноярский": "", "Красноярск": "",  "д.": "", "ул.": "улица", "пер.": "",
                "Дом ": " ", "Строение": "", "2-Я": "2-я", "Им.": "Имени ", "Корпус": "", " Г. ": " "}

for old, new in expressions.items():
    data["address_clean"] = data["address_clean"].str.replace(old, new, regex=False)

In [35]:
data["address_clean"] = "Красноярск, " + data["address_clean"]

In [36]:
exceptional_names = {" няя ": " Крайняя ", "Газеты  Рабочий": "Газеты Красноярский Рабочий","Проспект Мира 19  1": "Проспект Мира 19",
                     "улица Елены Стасовой 80 С 1": "улица Елены Стасовой 80"}

for old, new in exceptional_names.items():
    data["address_clean"] = data["address_clean"].str.replace(old, new, regex=False)

In [37]:
data["address_clean"] = data["address_clean"].str.rsplit(n=1).str.join(", ")

In [38]:
geolocator = Nominatim(user_agent="task_1_(xlsx->locations_dataset)", timeout=10)
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1, max_retries=2, error_wait_seconds=10, swallow_exceptions=True)
tqdm.pandas(desc="Геокодирование адресов")
data["geocoding_quality"] = data["address_clean"].progress_apply(geocode)

Геокодирование адресов: 100%|██████████| 420/420 [07:04<00:00,  1.01s/it]


In [39]:
data["latitude"] = data["geocoding_quality"].apply(lambda x: x.latitude if x else None)
data["longitude"] = data["geocoding_quality"].apply(lambda x: x.longitude if x else None)

Почему-то улицы Академгородок нет в Nominatim, поэтому руками добавляем все ПВЗ на ней

In [44]:
def add_akademgorodok_to_data(index: int, latitude: float, longitude: float, house_number: int) -> None:
    data.loc[index, ["latitude", "longitude", "geocoding_quality"]] = [latitude, longitude,
                    ", ".join((str(house_number), "улица Академгородок, район Академгородок, Красноярск",
                     "городской округ Красноярск, Красноярский край, Сибирский федеральный округ, 660000, Россия"))]

add_akademgorodok_to_data(index=4, latitude=55.987347, longitude=92.781677, house_number=74)
add_akademgorodok_to_data(index=12, latitude=55.985685, longitude=92.752050, house_number=12)
add_akademgorodok_to_data(index=321, latitude=55.987382, longitude=92.774715, house_number=66)
add_akademgorodok_to_data(index=324, latitude=55.992835, longitude=92.757799, house_number=17)

In [50]:
data = data.reindex(columns=["address_raw", "address_clean", "latitude", "longitude", "working_hours", "geocoding_quality"])
data.to_csv("pvz_krasnoyarsk.csv")

In [51]:
map = folium.Map(location=[data['latitude'].mean(), data['longitude'].mean()], zoom_start=11)
marker_cluster = MarkerCluster().add_to(map)
for _, row in data.iterrows():
    popup_text = f"<b>Адрес:</b> {row['address_clean']}<br> <b>Часы работы:</b> {row['working_hours']}<br> <b>Координаты:</b> {row['latitude']:.6f}, {row['longitude']:.6f}"
    folium.Marker(location=[row['latitude'], row['longitude']], popup=folium.Popup(popup_text, max_width=300), tooltip=row['address_clean']).add_to(marker_cluster)

map.save('pvz_krasnoyarsk_map.html')